# The Single Test Run: the Frozen Model on the Proof Period

## Natural-language request

> Now run the chosen model on the test set and check accuracy, then also the cost saved, and whether the guardrail metric was breached or not.

This is the contract's **proof** run (arrivals 1 January to 31 August 2017), made **once**, with settings frozen beforehand (project rule 2).

**Frozen before any test file was opened** (`models/frozen_choice.json` and `docs/DECISIONS.md`, 2026-11-18):

| Setting | Value |
|---|---|
| Method | **CatBoost**, fitted on train only (notebook 11), inputs prepared by the train-fitted preprocessor (notebook 10) |
| Rule | k = max(0, ⌊expected cancellations⌋ − buffer) |
| Buffers | **City Hotel 6, Resort Hotel 3** |
| Benchmark | Flat rule: train hotel rates (City 41.7%, Resort 25.2%), buffers **City 20, Resort 4** |
| Guardrail | At most **10 walked guests per 1,000 arrivals**, group-wide |
| Also reported | The shared-buffer version from notebook 12 (CatBoost 5, flat rule 16), disclosed as secondary |

## The order this notebook enforces

1. **Check the frozen settings:** the model and preprocessor files must be byte-for-byte the ones fingerprinted at the freeze.
2. **Predict from the test inputs only**, and save the predictions.
3. **Only then open the test labels**, and score.

If the frozen settings are ever changed and this notebook rerun, it stops: a change after seeing test results must be disclosed and both runs reported (`docs/EVALUATION_CONTRACT.md`).

**What this run can and cannot show.** It gives the honest test figures for the frozen choice: accuracy, euros, empty rooms, walks and the guardrail. It cannot make the forecast a live morning forecast (project rule 7: the book scored is the final book), and it does not test the vendor's accuracy claim.

## Step 1: check the frozen settings

In [1]:
from pathlib import Path
import hashlib
import json
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score

# Find the repository so this works from the repo folder or notebooks/.
repo_root = None
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "src" / "contract.py").exists():
        repo_root = folder
        break
if repo_root is None:
    raise FileNotFoundError("Could not find the project src/ folder.")
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import contract


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


freeze_path = repo_root / "models" / "frozen_choice.json"
freeze = json.loads(freeze_path.read_text(encoding="utf-8"))
freeze_fingerprint = sha256(freeze_path)
assert freeze["frozen_before_test"]
assert sha256(repo_root / freeze["model_file"]) == freeze["model_sha256"], "the model file changed after the freeze"
assert sha256(repo_root / freeze["preprocessor_file"]) == freeze["preprocessor_sha256"], "the preprocessor changed after the freeze"

# Run-once guard: an earlier run must have used exactly these frozen settings.
model_dir = repo_root / "data" / "models"
run_record_path = model_dir / "test_run_record.json"
if run_record_path.exists():
    previous = json.loads(run_record_path.read_text(encoding="utf-8"))
    if previous["freeze_sha256"] != freeze_fingerprint:
        raise RuntimeError("The frozen settings changed after a test run. Disclose the change and report both runs; "
                           "do not overwrite the first run's results.")

split_dir = repo_root / "data" / "splits"
manifest = json.loads((split_dir / "split_columns.json").read_text(encoding="utf-8"))
inputs, label = manifest["inputs"], manifest["label"]
HOTELS = ["City Hotel", "Resort Hotel"]
assert contract.WALK_CEILING == freeze["walk_ceiling_per_1000_arrivals"] and contract.WALK_COST == freeze["walk_cost_eur"]

print(f"Frozen at {freeze['frozen_at']}: {freeze['chosen_method']}, buffers {freeze['primary']['chosen_buffers']}; "
      f"flat rule buffers {freeze['primary']['flat_rule_buffers']}")
print("Model and preprocessor match their fingerprints.")

Frozen at 2026-11-18T01:57:07: CatBoost, buffers {'City Hotel': 6, 'Resort Hotel': 3}; flat rule buffers {'City Hotel': 20, 'Resort Hotel': 4}
Model and preprocessor match their fingerprints.


## Step 2: predict from the test inputs, before opening any outcome

`test.csv` holds the inputs and the scoring columns but **no label** (notebook 08 saved the label separately). The model sees only the eight inputs. The predictions are saved **before** `test_labels.csv` is opened.

In [2]:
test = pd.read_csv(split_dir / "test.csv", index_col="row", parse_dates=["arrival_date"],
                   dtype={"agent": str, "company": str})
assert label not in test.columns, "test.csv must not contain the label"
assert test["arrival_date"].min() >= pd.Timestamp(contract.PROOF_START)

preprocessor = joblib.load(repo_root / freeze["preprocessor_file"])
model = joblib.load(repo_root / freeze["model_file"])
fingerprints = (joblib.hash(preprocessor), joblib.hash(model))

warnings.filterwarnings("ignore", message="Found unknown categories")  # unseen agents/companies -> "infrequent"
X_test = preprocessor.transform(test[inputs])
assert list(X_test.columns) == json.loads((repo_root / "data" / "preprocessed" / "feature_names.json")
                                          .read_text(encoding="utf-8"))["output_columns"]

rates = freeze["flat_rule_hotel_rates_from_train"]
predictions = pd.DataFrame({
    "CatBoost": model.predict_proba(X_test.to_numpy(dtype=float))[:, 1],
    "flat-rule reference": test["hotel"].map(rates).to_numpy(),
}, index=test.index)
assert fingerprints == (joblib.hash(preprocessor), joblib.hash(model)), "nothing may be refitted on test"
assert predictions.notna().all().all() and predictions.apply(lambda p: p.between(0, 1).all()).all()

predictions_path = model_dir / "test_predictions.csv"
predictions.to_csv(predictions_path, index_label="row")
predictions_sha = sha256(predictions_path)
print(f"Predicted {len(test):,} test bookings ({test['arrival_date'].min():%Y-%m-%d} to {test['arrival_date'].max():%Y-%m-%d}); "
      f"predictions saved before any outcome was opened.")

Predicted 40,687 test bookings (2017-01-01 to 2017-08-31); predictions saved before any outcome was opened.


## Step 3: open the test labels

The labels are matched to the predictions **by booking row number**, and the saved predictions are checked to be unchanged.

In [3]:
assert sha256(predictions_path) == predictions_sha
test_labels = pd.read_csv(split_dir / "test_labels.csv", index_col="row")
assert test_labels.index.equals(test.index), "labels and bookings must match row for row"
test = test.join(test_labels)
y_test = test[label]
print(f"Test: {len(test):,} bookings, {test.groupby(['hotel', 'arrival_date']).ngroups} hotel-nights, "
      f"actual cancellation rate {y_test.mean():.1%}")

Test: 40,687 bookings, 486 hotel-nights, actual cancellation rate 38.7%


## Step 4: accuracy of the booking-level forecasts

**Accuracy** is the share of bookings classified correctly when every booking with a probability of 50% or more is called a cancellation. It is easy to read but it is **not what the contract scores**: the oversell decision uses the **sum** of probabilities per night, not a yes/no per booking. So it is shown with three other measures and with a reference point, the accuracy of predicting "nobody cancels" for every booking.

Val figures are shown beside test, for comparison with what was expected.

In [4]:
val = pd.read_csv(split_dir / "val.csv", index_col="row")
val_probabilities = pd.read_csv(model_dir / "val_probabilities.csv", index_col="row").loc[val.index]


def scores(y, p):
    return {"accuracy (50% cut-off)": accuracy_score(y, p >= 0.5), "ROC AUC": roc_auc_score(y, p),
            "log loss": log_loss(y, p), "Brier score": brier_score_loss(y, p),
            "mean predicted rate": p.mean(), "actual rate": y.mean()}


accuracy_table = pd.DataFrame({
    ("val", "CatBoost"): scores(val[label], val_probabilities["CatBoost"].to_numpy()),
    ("val", "flat-rule reference"): scores(val[label], val_probabilities["flat-rule reference"].to_numpy()),
    ("test", "CatBoost"): scores(y_test, predictions["CatBoost"].to_numpy()),
    ("test", "flat-rule reference"): scores(y_test, predictions["flat-rule reference"].to_numpy()),
}).T
accuracy_table["accuracy of 'nobody cancels'"] = [1 - val[label].mean()] * 2 + [1 - y_test.mean()] * 2
display(accuracy_table.style.format({"accuracy (50% cut-off)": "{:.1%}", "accuracy of 'nobody cancels'": "{:.1%}",
                                     "ROC AUC": "{:.3f}", "log loss": "{:.4f}", "Brier score": "{:.4f}",
                                     "mean predicted rate": "{:.1%}", "actual rate": "{:.1%}"}))

print("Accuracy by hotel (test):")
display(pd.DataFrame({h: {m: accuracy_score(y_test[test["hotel"] == h], predictions.loc[test["hotel"] == h, m] >= 0.5)
                          for m in predictions} for h in HOTELS}).style.format("{:.1%}"))

Accuracy by hotel (test):


,City Hotel,Resort Hotel
CatBoost,60.7%,69.2%
flat-rule reference,57.5%,69.2%


## Step 5: calibration and nightly error on test

The contract adds the probabilities up per hotel-night, so what matters most is whether the **expected** cancellations match the **actual** ones night by night. Bias above zero means too many cancellations were expected (a risk of walks), below zero too few (a risk of empty rooms).

In [5]:
bands = np.linspace(0, 1, 11)
band_names = [f"{int(lo * 100)}-{int(hi * 100)}%" for lo, hi in zip(bands[:-1], bands[1:])]
p = predictions["CatBoost"].to_numpy()
by_band = (pd.DataFrame({"predicted": p, "actual": y_test.to_numpy(), "band": pd.cut(p, bands, labels=band_names, include_lowest=True)})
           .groupby("band", observed=False).agg(predicted=("predicted", "mean"), actual=("actual", "mean"), bookings=("actual", "size")))
print("CatBoost calibration on test (fixed probability bands):")
display(by_band.style.format({"predicted": "{:.1%}", "actual": "{:.1%}", "bookings": "{:,}"}))

night_tables = {m: contract.nightly_table(test[contract.REQUIRED], predictions[m]) for m in predictions}
rows = []
for name, table in night_tables.items():
    error = table["expected"] - table["cancelled"]
    for hotel in ["group", *HOTELS]:
        e = error if hotel == "group" else error.xs(hotel, level="hotel")
        rows.append({"hotel": hotel, "method": name, "hotel-nights": len(e), "bias (rooms)": e.mean(),
                     "mean absolute error (rooms)": e.abs().mean(), "spread (rooms)": e.std()})
display(pd.DataFrame(rows).set_index(["hotel", "method"]).style.format(
    {"bias (rooms)": "{:+.2f}", "mean absolute error (rooms)": "{:.2f}", "spread (rooms)": "{:.2f}"}))

CatBoost calibration on test (fixed probability bands):


,predicted,actual,bookings
band,,,
0-10%,5.8%,7.0%,"2,642"
10-20%,14.3%,19.7%,"9,928"
20-30%,27.2%,44.2%,"7,467"
30-40%,31.7%,46.2%,"15,404"
40-50%,45.3%,53.5%,"3,399"
50-60%,54.5%,75.0%,"1,698"
60-70%,66.3%,63.8%,138
70-80%,74.2%,100.0%,4
80-90%,86.3%,42.9%,7


,,hotel-nights,bias (rooms),mean absolute error (rooms),spread (rooms)
hotel,method,,,,
group,CatBoost,486,-9.64,11.80,15.38
City Hotel,CatBoost,243,-14.16,16.54,19.33
Resort Hotel,CatBoost,243,-5.13,7.06,7.71
group,flat-rule reference,486,-2.00,9.31,14.32
City Hotel,flat-rule reference,243,-0.95,12.29,18.67
Resort Hotel,flat-rule reference,243,-3.04,6.33,7.77


## Step 6: cost, saving and guardrail at the frozen buffers

Each method's oversell count per hotel-night, from its frozen buffers, scored with `contract.night_cost`: euros, empty rooms, walked guests and walks per 1,000 arrivals, side by side (project rule 5).

- **Saving** = the flat rule's cost minus CatBoost's cost, on the **same hotel-nights**. Positive means CatBoost cost less.
- **Guardrail** = at most 10 walks per 1,000 arrivals for the **group** (project rule 6). Each hotel's rate is reported too, but there is no per-hotel threshold to judge it against.

In [6]:
def score(method, buffers):
    table = night_tables[method]
    buffer = np.where(table.index.get_level_values("hotel") == "City Hotel", buffers["City Hotel"], buffers["Resort Hotel"])
    k = pd.Series(np.maximum(np.floor(table["expected"].to_numpy()) - buffer, 0), index=table.index)
    out = {"group": contract.night_cost(table, k)}
    for hotel in HOTELS:
        part = table.xs(hotel, level="hotel", drop_level=False)
        out[hotel] = contract.night_cost(part, k.loc[part.index])
    return out


def cost_table(model_buffers, flat_buffers):
    results = {"CatBoost": score("CatBoost", model_buffers), "flat rule": score("flat-rule reference", flat_buffers)}
    rows = []
    for hotel in ["group", *HOTELS]:
        for name in ["flat rule", "CatBoost"]:
            r = results[name][hotel]
            rows.append({"hotel": hotel, "method": name,
                         "buffer": (model_buffers if name == "CatBoost" else flat_buffers)[hotel] if hotel != "group" else "",
                         "cost (EUR)": r["cost_eur"], "empty rooms": r["empty_rooms"], "walked guests": r["walked_guests"],
                         "walks per 1,000": r["walks_per_1000"],
                         "guardrail (<= 10, group)": ("met" if r["ceiling_met"] else "BREACHED") if hotel == "group"
                         else "no per-hotel threshold", "hotel-nights": r["nights"]})
    table = pd.DataFrame(rows).set_index(["hotel", "method"])
    saving = {h: results["flat rule"][h]["cost_eur"] - results["CatBoost"][h]["cost_eur"] for h in ["group", *HOTELS]}
    savings = pd.DataFrame({"saving vs flat rule (EUR)": saving,
                            "saving vs flat rule (%)": {h: saving[h] / results["flat rule"][h]["cost_eur"] for h in saving}})
    return table, savings, results


FMT = {"cost (EUR)": "{:,.0f}", "walks per 1,000": "{:.2f}"}
primary = freeze["primary"]
primary_table, primary_savings, primary_results = cost_table(primary["chosen_buffers"], primary["flat_rule_buffers"])
print("PRIMARY (frozen): separate buffer per hotel")
display(primary_table.style.format(FMT))
display(primary_savings.style.format({"saving vs flat rule (EUR)": "{:+,.0f}", "saving vs flat rule (%)": "{:+.1%}"}))

secondary = freeze["secondary_reported_for_transparency"]
both = lambda b: {"City Hotel": b, "Resort Hotel": b}
secondary_table, secondary_savings, secondary_results = cost_table(both(secondary["chosen_buffer"]), both(secondary["flat_rule_buffer"]))
print("SECONDARY (disclosed): one shared buffer for both hotels, which handicaps the flat rule (notebook 13, Part 5)")
display(secondary_table.style.format(FMT))
display(secondary_savings.style.format({"saving vs flat rule (EUR)": "{:+,.0f}", "saving vs flat rule (%)": "{:+.1%}"}))

PRIMARY (frozen): separate buffer per hotel


,saving vs flat rule (EUR),saving vs flat rule (%)
group,"-150,459",-4.7%
City Hotel,"+18,259",+0.9%
Resort Hotel,"-168,718",-13.3%


SECONDARY (disclosed): one shared buffer for both hotels, which handicaps the flat rule (notebook 13, Part 5)


,saving vs flat rule (EUR),saving vs flat rule (%)
group,"+431,799",+11.0%
City Hotel,"-206,104",-12.6%
Resort Hotel,"+637,903",+27.8%


## Step 7: month by month (primary settings)

Whether the result is steady across the eight test months or driven by a few.

In [7]:
def nightly_cost(method, buffers):
    table = night_tables[method]
    buffer = np.where(table.index.get_level_values("hotel") == "City Hotel", buffers["City Hotel"], buffers["Resort Hotel"])
    k = np.maximum(np.floor(table["expected"].to_numpy()) - buffer, 0)
    cost = ((table["cancelled"] - k).clip(lower=0) * table["stay_value"] + (k - table["cancelled"]).clip(lower=0) * contract.WALK_COST)
    walks = (k - table["cancelled"]).clip(lower=0)
    return cost, walks, table


rows = []
for name, method, buffers in [("flat rule", "flat-rule reference", primary["flat_rule_buffers"]),
                              ("CatBoost", "CatBoost", primary["chosen_buffers"])]:
    cost, walks, table = nightly_cost(method, buffers)
    month = table.index.get_level_values("arrival_date").to_period("M")
    arrivals = (table["book"] - table["cancelled"]).groupby(month).sum()
    for m, c in cost.groupby(month).sum().items():
        rows.append({"month": str(m), "method": name, "cost (EUR)": c,
                     "walks per 1,000": 1000 * walks.groupby(month).sum()[m] / arrivals[m]})
monthly = pd.DataFrame(rows).pivot(index="month", columns="method")
monthly[("saving (EUR)", "CatBoost vs flat")] = monthly[("cost (EUR)", "flat rule")] - monthly[("cost (EUR)", "CatBoost")]
display(monthly.style.format("{:,.0f}", subset=[c for c in monthly.columns if c[0] != "walks per 1,000"])
        .format("{:.2f}", subset=[c for c in monthly.columns if c[0] == "walks per 1,000"]))
print(f"CatBoost cheaper than the flat rule in {(monthly[('saving (EUR)', 'CatBoost vs flat')] > 0).sum()} of {len(monthly)} months.")

CatBoost cheaper than the flat rule in 3 of 8 months.


## Step 8: save the results and the run record

The run record ties these results to the exact frozen settings (by fingerprint), so a later change cannot silently replace them.

In [8]:
primary_table.to_csv(model_dir / "test_results_primary.csv")
primary_savings.to_csv(model_dir / "test_savings_primary.csv", index_label="hotel")
secondary_table.to_csv(model_dir / "test_results_secondary.csv")
accuracy_table.to_csv(model_dir / "test_accuracy.csv")
group = primary_results["CatBoost"]["group"]
run_record_path.write_text(json.dumps({
    "freeze_file": "models/frozen_choice.json", "freeze_sha256": freeze_fingerprint,
    "predictions_sha256": predictions_sha,
    "primary": {"catboost": primary_results["CatBoost"]["group"], "flat_rule": primary_results["flat rule"]["group"],
                "saving_eur": float(primary_savings.loc["group", "saving vs flat rule (EUR)"])},
}, indent=2) + "\n", encoding="utf-8")
for name in ["test_predictions.csv", "test_results_primary.csv", "test_savings_primary.csv", "test_results_secondary.csv",
             "test_accuracy.csv", "test_run_record.json"]:
    print(f"Saved data/models/{name}")

Saved data/models/test_predictions.csv
Saved data/models/test_results_primary.csv
Saved data/models/test_savings_primary.csv
Saved data/models/test_results_secondary.csv
Saved data/models/test_accuracy.csv
Saved data/models/test_run_record.json


## Step 9: the verdict in plain words

Generated from the numbers above, so it always matches them.

In [9]:
cat, flat = primary_results["CatBoost"], primary_results["flat rule"]
saving = primary_savings.loc["group", "saving vs flat rule (EUR)"]
share = primary_savings.loc["group", "saving vs flat rule (%)"]
acc = accuracy_table.loc[("test", "CatBoost")]

lines = [
    f"Accuracy: CatBoost classified {acc['accuracy (50% cut-off)']:.1%} of test bookings correctly at a 50% cut-off "
    f"(predicting 'nobody cancels' would score {accuracy_table.loc[('test', 'CatBoost'), 'accuracy of ' + chr(39) + 'nobody cancels' + chr(39)]:.1%}); "
    f"ROC AUC {acc['ROC AUC']:.3f}. It predicted a {acc['mean predicted rate']:.1%} cancellation rate against an actual {acc['actual rate']:.1%}.",
    (f"Cost: CatBoost cost EUR {cat['group']['cost_eur']:,.0f} against EUR {flat['group']['cost_eur']:,.0f} for the flat rule "
     f"with its own per-hotel buffers: " + (f"a saving of EUR {saving:,.0f} ({share:.1%})." if saving > 0
                                           else f"EUR {-saving:,.0f} MORE ({-share:.1%}); no saving.")),
    f"Rooms: {cat['group']['empty_rooms']:,} empty rooms and {cat['group']['walked_guests']:,} walked guests for CatBoost; "
    f"{flat['group']['empty_rooms']:,} and {flat['group']['walked_guests']:,} for the flat rule.",
    f"Guardrail (group, at most 10 walks per 1,000 arrivals): CatBoost {cat['group']['walks_per_1000']:.2f} -> "
    f"{'MET' if cat['group']['ceiling_met'] else 'BREACHED'}; flat rule {flat['group']['walks_per_1000']:.2f} -> "
    f"{'MET' if flat['group']['ceiling_met'] else 'BREACHED'}.",
    "By hotel (no per-hotel threshold given): " + "; ".join(
        f"{h}: CatBoost {cat[h]['walks_per_1000']:.2f}, flat rule {flat[h]['walks_per_1000']:.2f} walks per 1,000, "
        f"saving EUR {primary_savings.loc[h, 'saving vs flat rule (EUR)']:+,.0f}" for h in HOTELS) + ".",
    f"Secondary (shared buffer, flatters the model): saving EUR {secondary_savings.loc['group', 'saving vs flat rule (EUR)']:+,.0f} "
    f"({secondary_savings.loc['group', 'saving vs flat rule (%)']:+.1%}); CatBoost guardrail "
    f"{'met' if secondary_results['CatBoost']['group']['ceiling_met'] else 'BREACHED'} "
    f"({secondary_results['CatBoost']['group']['walks_per_1000']:.2f}).",
]
for line in lines:
    print("-", line)

- Accuracy: CatBoost classified 63.5% of test bookings correctly at a 50% cut-off (predicting 'nobody cancels' would score 61.3%); ROC AUC 0.679. It predicted a 27.2% cancellation rate against an actual 38.7%.
- Cost: CatBoost cost EUR 3,365,371 against EUR 3,214,912 for the flat rule with its own per-hotel buffers: EUR 150,459 MORE (4.7%); no saving.
- Rooms: 7,322 empty rooms and 210 walked guests for CatBoost; 7,166 and 179 for the flat rule.
- Guardrail (group, at most 10 walks per 1,000 arrivals): CatBoost 8.42 -> MET; flat rule 7.18 -> MET.
- By hotel (no per-hotel threshold given): City Hotel: CatBoost 7.33, flat rule 4.80 walks per 1,000, saving EUR +18,259; Resort Hotel: CatBoost 10.30, flat rule 11.29 walks per 1,000, saving EUR -168,718.
- Secondary (shared buffer, flatters the model): saving EUR +431,799 (+11.0%); CatBoost guardrail met (7.62).


## What these results do and do not establish

- **They are the honest proof figures for the frozen choice.** Nothing was tuned on test, and the choice was recorded before test was opened.
- **One period, one run.** Eight months of 2017 is one sample. A different year could come out differently, and the choose period already showed the model and the flat rule were close (notebook 13).
- **Historical, final-book replay** (project rule 7): every booking for a night is treated as on the book. A live morning forecast needs timestamped reservation snapshots.
- **Four provisional inputs** (`is_repeated_guest`, `previous_cancellations`, `previous_bookings_not_canceled`, `company`) have unverified timing. If any is shown to record information from after the forecast morning, these results must be redone without it.
- **Accuracy is not the contract's measure**, and nothing here validates the vendor's 95% accuracy claim, which needs the vendor's own definition and data.
- **Hotel walk rates are reported, not judged**: the client has given only a group-wide ceiling (project rule 6).